In [29]:
import pandas as pd
import numpy as np
import os

# read data
os.makedirs("../data/raw", exist_ok = True)
# data_url = "tbd"
raw_data = pd.read_csv("../data/raw/2019-Nov.csv")

print(raw_data.shape)
print(raw_data.info())
print(raw_data.head())

(67501979, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 4.5+ GB
None
                event_time event_type  product_id          category_id  \
0  2019-11-01 00:00:00 UTC       view     1003461  2053013555631882655   
1  2019-11-01 00:00:00 UTC       view     5000088  2053013566100866035   
2  2019-11-01 00:00:01 UTC       view    17302664  2053013553853497655   
3  2019-11-01 00:00:01 UTC       view     3601530  2053013563810775923   
4  2019-11-01 00:00:01 UTC       view     1004775  2053013555631882655   

               category_code   brand   price    user

In [30]:
# due to the larget dataset size, taking 20% random user for analysis
unique_users = raw_data['user_id'].unique()
print("Full dataset user counts:", len(unique_users))

Full dataset user counts: 3696117


In [31]:
np.random.seed(123)

sample_users = np.random.choice(unique_users, size = int(len(unique_users) * 0.2), replace = False)
print("Sample dataset user counts:", len(sample_users))

Sample dataset user counts: 739223


In [32]:
raw_data_sample = raw_data[raw_data['user_id'].isin(sample_users)].copy()
raw_data = raw_data_sample

raw_data.shape

(13542830, 9)

In [33]:
# transform event date to datetime and extract the date only
raw_data['event_time'] = pd.to_datetime(raw_data['event_time'], utc = True)

# check if there's any unreasonable price (negative price)
(raw_data["price"] < 0).any()

# remove duplicates
raw_data = raw_data.drop_duplicates()

In [ ]:
# Since currernt data is event-level, we need to transform the data into various data level based on the analysis we want to do.

# session-level analysis table
raw_data['purchase_revenue'] = raw_data['price'] * (raw_data['event_type'] == 'purchase')  

session_df = raw_data.groupby(['user_id', 'user_session']).agg(
    session_start = ('event_time', 'min'),
    session_end = ('event_time', 'max'),
    total_events = ('event_type', 'count'),
    view_count = ('event_type', lambda x: (x == 'view').sum()),
    cart_count = ('event_type', lambda x: (x == 'cart').sum()),
    purchase_count = ('event_type', lambda x: (x == 'purchase').sum()),
    total_revenue = ('purchase_revenue', 'sum')
    ).reset_index()

# session action
session_df['has_view'] = (session_df['view_count'] > 0).astype(int)
session_df['has_cart'] = (session_df['cart_count'] > 0).astype(int)
session_df['has_purchase'] = (session_df['purchase_count'] > 0).astype(int)

# session duration
session_df['session_duration_sec'] = (
    session_df['session_end'] - session_df['session_start']
).dt.total_seconds()

session_df[session_df['has_purchase'] > 0].head()

In [35]:
# user-level analysis table
user_df = (session_df.groupby('user_id').agg(
    total_sessions = ('user_session', 'nunique'),
    total_views = ('view_count', 'sum'),
    total_carts = ('cart_count', 'sum'),
    total_purchases = ('purchase_count', 'sum'),
    total_revenue = ('total_revenue', 'sum')
    )
    ).reset_index()

user_df[user_df['total_sessions'] > 1].head()

# adding if user has purchase with 1 = true, 0 = false
user_df['has_purchase'] = (user_df['total_purchases'] > 0).astype(int)

In [36]:
# funnel table
funnel = pd.DataFrame(
    {
        'stage' : ['view', 'cart', 'purchase'],
        'sessions' : [
            session_df['has_view'].sum(),  # counts of sessions which has view
            session_df['has_cart'].sum(),  # counts of sessions which has cart
            session_df['has_purchase'].sum()  # counts of purchase which has purchase
        ]
    }
)

print(funnel)
# the ['sessions'] here is also funnel volume

      stage  sessions
0      view   2781434
1      cart    348312
2  purchase    153412


In [ ]:
# export analysis tables

# os.makedirs("../data/processed", exist_ok=True)

raw_data.to_csv("../data/analysis_table/data_cleaned.csv", index = False)
session_df.to_csv("../data/analysis_table/session_table.csv", index = False)
user_df.to_csv("../data/analysis_table/user_table.csv", index = False)
funnel.to_csv("../data/analysis_table/funnel_table.csv", index = False)